# Advanced Baselines for Entity Resolution

This notebook provides an interactive interface for running entity resolution experiments.

**Experiments covered:**
1. Model upgrade (GPT-5.2-pro)
2. Reasoning token analysis (none → xhigh)
3. In-context learning (few-shot)
4. SFT dataset creation

**Usage:**
- Use dry-run mode to preview experiments without API calls
- Start with small subsets (limit=10-100) to validate
- Run full experiments when ready

In [ ]:
# Setup: Add project root to path
import sys
sys.path.append("..")

# Core imports
from scripts.baselines.config import ExperimentConfig, CONFIGS
from scripts.baselines.example_selector import get_examples, print_example_summary
from scripts.evaluate import evaluate, print_evaluation_report

# Check available configs
print("Available predefined configurations:")
for name in sorted(CONFIGS.keys()):
    config = CONFIGS[name]
    print(f"  {name}: model={config.model}, reasoning={config.reasoning_effort}, examples={config.n_examples}")

## 1. Configuration Preview

Preview experiment configurations before running.

In [ ]:
# Create a custom configuration
config = ExperimentConfig(
    model="gpt-5.2-pro",
    reasoning_effort="medium",
    n_examples=0,  # 0 = zero-shot
    parallel=10,
    dry_run=True,  # Preview only
)

print("Configuration:")
print(config.describe())

print("\nCost estimate (800 pairs):")
cost = config.estimate_cost(800)
print(f"  Input tokens: {cost['input_tokens']:,}")
print(f"  Output tokens: {cost['output_tokens']:,}")
print(f"  Estimated cost: ${cost['estimated_cost_usd']}")
print(f"  Per pair: ${cost['cost_per_pair_usd']}")

## 2. Example Selection Preview

Preview examples for few-shot learning.

In [ ]:
# Get examples using diverse strategy
examples = get_examples(strategy="diverse", n=8)
print_example_summary(examples)

# Preview first example
print("\nFirst example:")
ex = examples[0]
print(f"  Left: {ex['left'].get('caption', 'Unknown')}")
print(f"  Right: {ex['right'].get('caption', 'Unknown')}")
print(f"  Label: {ex['judgement']}")

## 3. Zero-Shot Experiment (Dry Run)

Preview a zero-shot experiment without making API calls.

In [ ]:
from scripts.baselines.llm_zeroshot import run_experiment as run_zeroshot

# Dry run - preview prompts and cost
result = run_zeroshot(
    input_path="data/samples/sample_1000.json",
    output_dir="data/outputs",
    model="gpt-5-nano",
    reasoning_effort="medium",
    limit=5,  # Only preview 5 pairs
    offset=200,
    parallel=None  # Sequential for dry run
)

# Note: To actually run, remove limit or set parallel > 1

## 4. Few-Shot Experiment (Dry Run)

In [ ]:
from scripts.baselines.llm_fewshot import run_experiment as run_fewshot

# Few-shot dry run
config = ExperimentConfig(
    model="gpt-5-nano",
    reasoning_effort="medium",
    n_examples=8,
    example_strategy="diverse",
    limit=5,
    dry_run=True
)

result = run_fewshot(config)

## 5. Reasoning Sweep Preview

In [ ]:
from scripts.baselines.reasoning_sweep import run_reasoning_sweep, REASONING_LEVELS

# Preview reasoning sweep
print("Reasoning levels to test:", REASONING_LEVELS)

# Estimate costs for each level
print("\nCost estimates (800 pairs):")
for level in REASONING_LEVELS:
    config = ExperimentConfig(reasoning_effort=level)
    cost = config.estimate_cost(800)
    print(f"  {level}: ${cost['estimated_cost_usd']} ({cost['total_tokens']:,} tokens)")

In [ ]:
# Dry run of reasoning sweep
results = run_reasoning_sweep(
    model="gpt-5-nano",
    limit=5,  # Small subset
    dry_run=True  # Preview only
)

## 6. SFT Dataset Creation

In [ ]:
from scripts.finetune.prepare_sft_data import preview_examples, prepare_dataset, validate_dataset

# Preview what training examples look like
preview_examples("data/samples/sample_1000.json", n=2)

In [ ]:
# Create SFT dataset (requires 10K sample)
# First check if 10K sample exists
from pathlib import Path

sample_10k = Path("data/samples/sample_10000.json")
if sample_10k.exists():
    print("10K sample exists. Creating SFT dataset...")
    metadata = prepare_dataset(
        input_path="data/samples/sample_10000.json",
        output_dir="data/finetune",
        n_train=2000,
        n_val=200
    )
else:
    print("10K sample not found. Create it first with:")
    print("  python scripts/create_proper_sample.py --n 10000 --output data/samples/sample_10000.json")
    print("\nAlternatively, use the 1K sample with smaller training set:")
    metadata = prepare_dataset(
        input_path="data/samples/sample_1000.json",
        output_dir="data/finetune",
        n_train=150,  # 150 from 200 dev pairs
        n_val=30
    )

In [ ]:
# Validate created SFT dataset
results = validate_dataset("data/finetune")

if results["valid"]:
    print("Validation PASSED")
    print("\nFile statistics:")
    for key, count in results.get("stats", {}).items():
        print(f"  {key}: {count} examples")
else:
    print("Validation FAILED")
    for error in results["errors"]:
        print(f"  - {error}")

## 7. Running Actual Experiments

**Warning:** The cells below make actual API calls and incur costs.

Start with small limits (10-100 pairs) to validate, then scale up.

In [ ]:
# UNCOMMENT TO RUN ACTUAL EXPERIMENTS
# WARNING: This will make API calls and incur costs!

# # Small test run (10 pairs)
# from scripts.baselines.llm_zeroshot import run_experiment
# 
# metrics, results = run_experiment(
#     input_path="data/samples/sample_1000.json",
#     output_dir="data/outputs",
#     model="gpt-5-nano",
#     reasoning_effort="medium",
#     limit=10,
#     offset=200,
#     parallel=5
# )
# 
# print(f"\nF1: {metrics['f1']:.2%}")
# print(f"Accuracy: {metrics['accuracy']:.2%}")

In [ ]:
# UNCOMMENT TO RUN GPT-5.2-PRO EXPERIMENT
# WARNING: This is expensive (~$50-100 for 800 pairs)

# from scripts.baselines.llm_zeroshot import run_experiment
# 
# metrics, results = run_experiment(
#     input_path="data/samples/sample_1000.json",
#     output_dir="data/outputs",
#     model="gpt-5.2-pro",
#     reasoning_effort="medium",
#     offset=200,  # Skip dev set
#     parallel=10  # Lower parallelism for expensive model
# )

## 8. Compare Results

In [ ]:
from scripts.experiment import list_experiments, compare_experiments

# List recent experiments
print("Recent experiments:")
experiments = list_experiments(limit=10)
for exp in experiments:
    print(f"  {exp['run_id']}: F1={exp.get('metrics', {}).get('f1', 0):.2%}")

In [ ]:
# Compare all LLM experiments
compare_experiments(method="llm")